In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, input_file_name

spark = (
    SparkSession.builder
    .appName("Medallion Bronze")
    .master("local[*]")
    .getOrCreate()
)

print("========== SPARK APPLICATION INFO ==========")

print("Application ID   :", spark.sparkContext.applicationId)
print("Application Name :", spark.sparkContext.appName)
print("Spark Version    :", spark.version)
print("Master           :", spark.sparkContext.master)
print("Web UI           :", spark.sparkContext.uiWebUrl)
print("Default Parallel :", spark.sparkContext.defaultParallelism)
print("Default Partitons :",spark.conf.get("spark.sql.shuffle.partitions"))

print("============================================")
spark

========== SPARK APPLICATION INFO ==========
Application ID   : local-1792929573966
Application Name : Medallion Bronze
Spark Version    : 3.5.0
Master           : local[*]
Web UI           : http://Arjun:4043
Default Parallel : 18
Default Partitons : 200


In [5]:
from pyspark.sql.functions import (
    sum as spark_sum,
    countDistinct,
    round as spark_round
)

silver_df = spark.read.parquet(
    "D:/pyspark_vs_code/medallion_project_duckdb/data/silver"
)

gold_df = (
    silver_df
    .groupBy("state")
    .agg(
        spark_sum("sales").alias("total_sales"),
        countDistinct("order_id").alias("total_orders")
    )
    .withColumn(
        "total_sales",
        spark_round("total_sales", 2)
    )
)

gold_df.show()

+--------------------+-----------+------------+
|               state|total_sales|total_orders|
+--------------------+-----------+------------+
|                Utah|  168689.40|        1538|
|              Hawaii|   85340.69|         814|
|           Minnesota|  181497.34|        1590|
|                Ohio|  359566.70|        2981|
|              Oregon|  366316.69|        3005|
|               Texas| 1039762.89|        6406|
|        Pennsylvania|  178550.70|        1604|
|         Connecticut|  177010.04|        1599|
|            Nebraska|  177600.32|        1601|
|              Nevada|  182117.16|        1620|
|          Washington|  345998.70|        2947|
|            Illinois|  266545.40|        2273|
|            Oklahoma|   90169.20|         823|
|District of Columbia|   94050.73|         857|
|            Missouri|  257477.04|        2229|
|             Georgia|  174473.62|        1554|
|            Michigan|  186083.13|        1673|
|            Virginia|  268891.32|      

In [6]:
gold_df.write \
    .mode("overwrite") \
    .parquet("D:/pyspark_vs_code/medallion_project_duckdb/data/gold")